# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install `mlcroissant` if not already installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset's Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Retrieve the metadata as a dictionary
metadata = dataset.metadata.to_json()

print("Dataset Loaded:")
print(f"Title: {metadata.get('name')}")
print(f"Description: {metadata.get('description')}")


## 2. Data Overview
Explore the available record sets, and list their fields and column `@id`s. Reference entities by their `@id` fields for consistency.

Let's show all available record sets and their fields (if present) using the Croissant schema.

In [ ]:
# List all available record sets by @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- Record Set @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field']
            if isinstance(fields, dict):
                fields = [fields]
            print("  Fields:")
            for field in fields:
                print(f"    - Field @id: {field['@id']} (dataType: {field.get('dataType')})")
        if 'column' in rs:
            columns = rs['column']
            if isinstance(columns, dict):
                columns = [columns]
            print("  Columns:")
            for col in columns:
                print(f"    - Column @id: {col['@id']} (name: {col.get('name')})")
        print()
# For demonstration, print a preview of the first 1-2 records from a record set (if it exists)
if record_sets:
    rs_id = record_sets[0]['@id']
    print(f"First 2 records from record set {rs_id}:")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        pprint.pprint(record)
        if i >= 1:
            break

## 3. Data Extraction
Extract data from each record set into a pandas DataFrame. All references will use the `@id` fields of the record sets and columns above.

In [ ]:
# Prepare to load all record sets into pandas DataFrames
dfs = {}
if not record_sets:
    print("No record sets to extract data from.")
else:
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"Loading record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        # Only create DataFrames with records
        if records:
            dfs[rs_id] = pd.DataFrame(records)
            print(f"  DataFrame columns for {rs_id}: {dfs[rs_id].columns.tolist()}")
            display(dfs[rs_id].head())
        else:
            print(f"  No records found for record set {rs_id}.")
# For demonstrative EDA below, choose first available DataFrame
main_rs_id = next(iter(dfs), None)

## 4. Exploratory Data Analysis (EDA)
Perform sample EDA steps: filtering on a numeric field, normalization, and grouping by a categorical column.

*Adjust the field `@id`s and logic to match your available record set/field structure, referencing by `@id` only!*

In [ ]:
import numpy as np
# Ensure we have at least one record set DataFrame to analyze
if not main_rs_id:
    print("No DataFrames available for EDA.")
else:
    df = dfs[main_rs_id]
    print(f"Using record set {main_rs_id} for EDA.")
    # Try to select numeric columns -- fallback to generic field names if structure unknown
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_columns:
        numeric_field_id = numeric_columns[0]  # For demonstration, pick the first numeric column
        print(f"Sample numeric field: {numeric_field_id}")
    else:
        print("No numeric columns found. Aborting numeric EDA.")
        numeric_field_id = None

    if numeric_field_id:
        # Filter records where the selected numeric field > a threshold (use 0th percentile as placeholder if unknown)
        threshold = df[numeric_field_id].dropna().quantile(0.50) # Median as arbitrary threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered {filtered_df.shape[0]} records where {numeric_field_id} > {threshold:.2f}")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized values for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt to group by a categorical field (pick first string column different from numeric_field_id)
        cat_columns = df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        for col in cat_columns:
            if col != numeric_field_id:
                group_field = col
                break
        if group_field and group_field in filtered_df.columns:
            print(f"Grouping by categorical field: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print("Mean of numeric field per group:")
            display(grouped.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of a numeric field or the relationship between two fields in the record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Optionally plot the normalized values
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(7, 4))
        sns.histplot(filtered_df[norm_col], kde=True, color='orange')
        plt.title(f"Distribution of Normalized {numeric_field_id}")
        plt.xlabel(norm_col)
        plt.ylabel("Count")
        plt.show()

    # Scatterplot of numeric vs categorical (if found)
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: missing numeric field or data.")

## 6. Conclusion
- The FAIR^2 dataset for rangeland management adoption predictors was loaded via its Croissant schema using `mlcroissant`.
- Available record sets and fields were referenced and displayed by their `@id` fields for consistent, schema-driven analysis.
- Data was extracted and processed: numeric fields were normalized and grouped to demonstrate practical exploratory data analysis.
- Visualizations assisted in understanding underlying value distributions and group-level trends for select fields.

Further steps can include more detailed feature engineering and modeling for predicting adoption or for deeper policy/equity analysis as described in the dataset documentation.